# Phase 2 descriptive analysis

Regenerates the supplementary descriptive figures and summary tables of one Phase 2 study from its
`export-phase2` bundle **alone** (spec `specs/session-12-phase25-release.md` §5, §6).

* Input: a bundle written by `ehr-simulator export-phase2`. This notebook never opens SQLite,
  never asks for the pseudonym secret and never reads a keyfile.
* Every join, rate and threshold lives in `ehr_simulator.analysis`; this notebook only orchestrates.
* **Descriptive only.** No hypothesis test, interval or effect estimate. Results are split by
  `config_version` (never pooled; one figure per version) and grouped by the ITT arm of the
  realised activation.
* The figures are supplementary; the main message figures will follow the approved analysis plan.
* Privacy: no clinician name, free text, raw event payload or patient / clinician id is printed or
  plotted. Real cohort outputs need a manual disclosure review before they leave the analysis
  environment.

## 1. Environment and input

| Variable | Default | Meaning |
| --- | --- | --- |
| `EHR_SIM_PHASE2_BUNDLE` | `../tests/fixtures/analysis_bundle/walked` | validated export bundle (committed synthetic example) |
| `EHR_SIM_ANALYSIS_OUT` | `../analysis_out/phase2` | report directory (git ignored) |
| `EHR_SIM_ANALYSIS_OVERWRITE` | `refuse` | `replace` swaps an existing report atomically |

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from ehr_simulator import __version__
from ehr_simulator.analysis import load_phase2_bundle
from ehr_simulator.analysis.figures import build_figure
from ehr_simulator.analysis.report import MANIFEST_NAME, Overwrite, publish_report
from ehr_simulator.analysis.summary import build_summaries

BUNDLE = Path(os.environ.get("EHR_SIM_PHASE2_BUNDLE", "../tests/fixtures/analysis_bundle/walked"))
OUT = Path(os.environ.get("EHR_SIM_ANALYSIS_OUT", "../analysis_out/phase2"))
OVERWRITE = Overwrite(os.environ.get("EHR_SIM_ANALYSIS_OVERWRITE", Overwrite.REFUSE))

pd.set_option("display.max_rows", 60)
print(f"ehr_simulator {__version__}")
print(f"bundle: {BUNDLE}")
print(f"output: {OUT} (overwrite: {OVERWRITE})")

## 2. Export and manifest validation

`load_phase2_bundle` checks the manifest schema (`phase2_export_v1`), the declared file set, every
SHA256 and row count, the frozen headers, strict cell types, the configuration snapshots (hash
recomputed) and every key and join. Any failure stops here with `AnalysisError`; nothing is written.

In [ ]:
bundle = load_phase2_bundle(BUNDLE)
summaries = build_summaries(bundle)

print(f"study: {bundle.study_id}")
print(f"input manifest SHA256: {bundle.manifest_sha256}")
print(f"practice files declared (verified, excluded): {bundle.practice_declared}")
print(f"cohort: {summaries.cohort.value}")

## 3. Cohort and configuration provenance

Each configuration version is reported on its own. The same `question_id` in two versions is not
assumed to be the same question.

In [ ]:
pd.DataFrame(
    [
        {
            "config_version": v.version,
            "config_hash": v.config_hash,
            "activated_at": v.activated_at,
            "timepoints_minutes": v.timepoints_minutes,
            "questions": len(v.questions.questions),
        }
        for v in bundle.versions.values()
    ]
)

## 4. Case flow

Planned schedule items (generation version, planned arm) beside activated measured cases (activation
version, realised ITT arm). A planned item is never a case; replacements are counted on the incomplete
original, and an activated replacement is also a case of its own.

In [ ]:
summaries.case_flow

In [ ]:
for version in bundle.versions:  # one figure per config version
    display(build_figure("supplementary_02_case_flow", bundle, summaries, version))

## 5. Responses and divergence

Denominator: reached primary observations where the question applied. Hidden branches
(`not_applicable`) are never missing; rule (`auto_value`) answers are counted and plotted apart from
clinician answers (dotted lines, hollow markers). Free text is summarised by status only.

The S10 `newly_visible_data` annotations need the clinical source data, which the export does not
carry; they are therefore not part of this reproducible figure set (use `divergence-view` for
exploratory, adapter backed inspection). A case-level image of one explicitly selected patient is
available as `ehr_simulator.analysis.figures.case_divergence_figure`; it is never batch generated.

In [ ]:
summaries.response_provenance

In [ ]:
for version in bundle.versions:  # one figure per config version
    display(build_figure("supplementary_03_response_provenance", bundle, summaries, version))

In [ ]:
summaries.option_proportions

In [ ]:
summaries.numeric_responses

In [ ]:
for version in bundle.versions:  # one figure per config version
    display(build_figure("supplementary_01_divergence", bundle, summaries, version))

## 6. Timing and attention

Wall clock `elapsed_seconds` (S10) and browser `foreground` / `active` seconds (S11j) are distinct
measures. Foreground / active count only where `telemetry_status` is `complete`; any other status is
counted as excluded, never as 0 seconds.

In [ ]:
summaries.attention

In [ ]:
for version in bundle.versions:  # one figure per config version
    display(build_figure("supplementary_04_attention", bundle, summaries, version))

## 7. Intervention integrity and per protocol

Over reached primary observations, grouped by ITT arm (never by exposure). Delivered / viewed apply to
the AI arm; on the no AI arm viewing is *not applicable* and AI exposure is *leakage*. Unknown and
indeterminate values are counted, never read as no.

In [ ]:
summaries.intervention

In [ ]:
for version in bundle.versions:  # one figure per config version
    display(build_figure("supplementary_05_intervention", bundle, summaries, version))

In [ ]:
summaries.denominators

## 8. Output manifest

`publish_report` re-validates the bundle, then writes every figure once per config version
(`figures/<stem>_<version>`, SVG + PDF), the summary tables,
`denominators.csv` and `analysis_manifest.json` to a staging directory and publishes it atomically.
An existing report is refused unless `EHR_SIM_ANALYSIS_OVERWRITE=replace`.

In [ ]:
inventory = publish_report(BUNDLE, OUT, overwrite=OVERWRITE)
manifest = json.loads((inventory.out_dir / MANIFEST_NAME).read_text())
shown = ("input", "config_versions", "cohort", "figure_role", "settings")
print(json.dumps({k: manifest[k] for k in shown}, indent=2))
pd.DataFrame(manifest["figures"])

## Interpretation limitations

* Descriptive summaries only: no test, interval, effect estimate or composite divergence score. The
  figures are supplementary; the main message figures follow the approved statistical analysis
  plan.
* Versions are never pooled; a question changed between versions is a different question.
* Arms are the ITT arm of the realised activation. AI delivery failures and no AI leakage are shown
  separately and never move an observation to the other arm.
* Unknown is not zero: blank durations, unknown delivery or viewing and indeterminate per protocol
  status are counted as unknown.
* Small samples (as in the synthetic example) give no basis for uncertainty bands or comparisons.